# 空间信息增益

在训练、验证和测试事件序列上，按震级展示 空间 LL 差值（ST-NKF − ETAS）。红色表示 ST-NKF 的对应 LL 更高，蓝色表示 ETAS 更高。色标以差值绝对值的 90% 分位数对称截取。这是事件序列诊断图。

改编自 [zhancx-edu/NeuralKernelFunction 的 plot spatial information gain.ipynb](https://github.com/zhancx-edu/NeuralKernelFunction/blob/febc15aefc316cf128ea217bbd60321c88fbaab1/plot%20spatial%20information%20gain.ipynb)，适配本仓库 PyTorch 输出。

在仓库根目录启动 Jupyter，先安装 `requirements-plotting.txt`。修改下面的参数单元格后，从上到下执行。结果保存到 `fig/`；本 Notebook 不启动训练。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "run_models.py").is_file():
    raise FileNotFoundError("请在 ST-NKF-pytorch 仓库根目录启动 Jupyter / nbconvert。")

from matplotlib import colors

In [ ]:
DATASETS = ["SCEDC_20", "SCEDC_25", "SCEDC_30", "ComCat", "SaltonSea", "SanJac", "WHITE"]
CSV_DIR = ROOT / "csv"
FIG_DIR = ROOT / "fig"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300
SHOW_FIGURES = True

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.linewidth": 0.8})

In [ ]:
MODEL_TAGS = {"ST-NKF": "neural_neural_neural", "ETAS": "empirical_empirical_empirical"}

def load_pair(dataset):
    frames = {}
    identity = ["step", "time_days", "magnitude", "x", "y"]
    required = identity + ["logli_t", "logli_s"]
    for model, tag in MODEL_TAGS.items():
        path = CSV_DIR / f"{dataset}_{tag}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；请先分别运行该目录的 ETAS 和 ST-NKF。")
        data = pd.read_csv(path)
        missing = set(required) - set(data.columns)
        if missing:
            raise ValueError(f"{path}: 缺少列 {sorted(missing)}")
        data = data.loc[data["step"].isin(["train", "val", "test"])].reset_index(drop=True)
        if data.empty:
            raise ValueError(f"{path}: 没有 train / val / test 事件。")
        numeric = ["time_days", "magnitude", "x", "y", "logli_t", "logli_s"]
        if not np.isfinite(data[numeric].to_numpy(dtype=float)).all():
            raise ValueError(f"{path}: 事件或似然列包含非有限值。")
        order = data["step"].map({"train": 0, "val": 1, "test": 2}).to_numpy()
        if np.any(np.diff(order) < 0):
            raise ValueError(f"{path}: 事件未按 train、val、test 顺序排列。")
        frames[model] = data
    nkf, etas = frames["ST-NKF"], frames["ETAS"]
    if not nkf[identity].equals(etas[identity]):
        raise ValueError(f"{dataset}: 两个 CSV 的事件或切分不一致，不能逐行比较。")
    return nkf, etas

def save_figure(fig, stem, pdf=False):
    fig.savefig(FIG_DIR / (stem + ".png"), dpi=DPI, bbox_inches="tight")
    if pdf:
        fig.savefig(FIG_DIR / (stem + ".pdf"), bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)
    print("Saved:", FIG_DIR / (stem + ".png"))

def plot_information_gain(nkf, etas, dataset):
    x = np.arange(1, len(nkf) + 1)
    gain = (nkf["logli_s"] - etas["logli_s"]).to_numpy()
    vmax = max(float(np.percentile(np.abs(gain), 90)), 1e-12)
    norm = colors.Normalize(vmin=-vmax, vmax=vmax)
    fig, ax = plt.subplots(figsize=(12, 5))
    scatter = ax.scatter(x, nkf["magnitude"], c=gain, cmap="coolwarm", norm=norm, s=2)
    counts = [int(nkf["step"].eq(step).sum()) for step in ("train", "val", "test")]
    boundaries = np.cumsum([0] + counts)
    for boundary in boundaries[1:-1]:
        ax.axvline(boundary + 0.5, color="grey", ls="--", lw=0.8)
    for i, label in enumerate(("Training", "Validation", "Test")):
        if counts[i]:
            ax.text((boundaries[i] + boundaries[i + 1] + 1) / 2, 0.97,
                    label, transform=ax.get_xaxis_transform(), ha="center", va="top",
                    bbox={"facecolor": "white", "edgecolor": "grey", "alpha": 0.8})
    ax.set_xlabel("Index of earthquakes")
    ax.set_ylabel("Magnitude")
    ax.set_title(dataset)
    ax.grid(True, ls="--", alpha=0.3)
    ax.spines["right"].set_visible(False)

    ax.spines["top"].set_visible(False)

    cbar = fig.colorbar(scatter, ax=ax, pad=0.03, aspect=20, extend="both")
    cbar.set_label("Spatial log-likelihood: ST-NKF − ETAS")
    fig.tight_layout()
    save_figure(fig, f"IG-{dataset}_spatial-Magnitude")

In [ ]:
for dataset in DATASETS:
    nkf, etas = load_pair(dataset)
    print(f"{dataset}: train={nkf['step'].eq('train').sum()}, "
          f"val={nkf['step'].eq('val').sum()}, test={nkf['step'].eq('test').sum()}")
    plot_information_gain(nkf, etas, dataset)